In [ ]:
import pandas as pd
import numpy as np

import itertools


In [ ]:
inFileName = 'flip.csv'

inDf = pd.read_csv(inFileName) #this is for Appendix D Sheet

In [ ]:
inDf[10750:10756]

,a,b,c,d
10750,Point Name: CHILLER.ZONE:STRT DRTN,NaN,NaN,NaN
10751,Point Type: LAO,NaN,NaN,NaN
10752,Supervised: Yes,NaN,NaN,NaN
10753,Revision Number: 3,NaN,NaN,NaN
10754,Classification: Building Automation ...,NaN,NaN,NaN
10755,Descriptor: STRT DRTN,NaN,NaN,NaN


In [ ]:
inDf['new_column'] = inDf[['a', 'b', 'c', 'd']].astype(str).agg(' '.join, axis=1)

In [ ]:
inDf = inDf.drop(['a','b','c','d'], axis = 1)

In [ ]:
inDf[10750:10756]

,new_column
10750,Point Name: CHILLER.ZONE:STRT DRT...
10751,Point Type: LAO nan n...
10752,Supervised: Yes nan nan nan
10753,Revision Number: 3 nan nan nan
10754,Classification: Building Automation ...
10755,Descriptor: STRT DRTN nan nan nan


In [ ]:
for count, row in enumerate(inDf['new_column']):
    #print(row)
    inDf['new_column'][count] = row.replace('nan nan nan','').strip()


for count, row in enumerate(inDf['new_column']):
    #print(row)
    if 'Actuator Type' in row:

        inDf['new_column'][count] = row.replace('Actuator Type','Actuator Type:')

In [ ]:
inDf[['Attribute', 'Value']] = inDf['new_column'].str.split(':', 1, expand=True)

In [ ]:
inDf[10750:10756]

,new_column,Attribute,Value
10750,Point Name: CHILLER.ZONE:STRT DRTN,Point Name,CHILLER.ZONE:STRT DRTN
10751,Point Type: LAO,Point Type,LAO
10752,Supervised: Yes,Supervised,Yes
10753,Revision Number: 3,Revision Number,3
10754,Classification: Building Automation,Classification,Building Automation
10755,Descriptor: STRT DRTN,Descriptor,STRT DRTN


In [ ]:
vals = []
atts = []
skipTwo = 0

for count, (full, val, att) in enumerate(zip(inDf['new_column'], inDf['Value'], inDf['Attribute'])):
    #print(att)
    if 'Offset' in att:
        #print(att)
        skipTwo = 3
        #print(count,inDf['Attribute'][count+1],inDf['Attribute'][count+2])
        for at in att.split('   '):

            if len(at) > 2:

                atts.append(at.strip())

        for at1, at2 in zip(inDf['Attribute'][count+1].split('\t'), inDf['Attribute'][count+2].split('\t')):

            if len(at1) > 1 and len(at2) > 1:

                vals.append(at1.strip() + ', ' + at2.strip())

    if skipTwo == 0:

        atts.append(att)
        vals.append(val)

    else:

        skipTwo -= 1



In [ ]:
inDf[10750:10756]

,new_column,Attribute,Value
10750,Point Name: CHILLER.ZONE:STRT DRTN,Point Name,CHILLER.ZONE:STRT DRTN
10751,Point Type: LAO,Point Type,LAO
10752,Supervised: Yes,Supervised,Yes
10753,Revision Number: 3,Revision Number,3
10754,Classification: Building Automation,Classification,Building Automation
10755,Descriptor: STRT DRTN,Descriptor,STRT DRTN


In [ ]:
outDf = pd.DataFrame(columns =['Attributes', 'Values'])

In [ ]:
outDf['Attributes'] = atts
outDf['Values'] = vals


In [ ]:
vals = []
atts = []
skip = 0


for count, (val, att) in enumerate(zip(outDf['Values'], outDf['Attributes'])):

    if val == None:

        if att.find('*') != 0 and att != 'nan':

            #print (count, att, val)
            continue

    if val == '' or att == 'Text Table':
        crit = True
        skip = 1
        conc = outDf['Values'][count]


        while crit is True:

            if outDf['Values'][count+skip] == None and outDf['Attributes'][count+skip].find('*') != 0 and outDf['Attributes'][count+skip] != 'nan':
                #print(count, att, outDf['Values'][count+skip])
                conc += ', ' + outDf['Attributes'][count+skip]

                skip +=1

            else:

                crit = False

                vals.append(conc)
                atts.append(att)

    if skip <= 1:

        atts.append(att)
        vals.append(val)

    else:

        skip -=1


In [ ]:
outDf.head(5)

,Attributes,Values
0,Point System Name,AHU1.ZONE:ACT CLG STPT
1,Point Name,AHU1.ZONE:ACT CLG STPT
2,Point Type,LAO
3,Supervised,Yes
4,Revision Number,3


In [ ]:
outDf2 = pd.DataFrame(columns =['Attributes', 'Values'])

outDf2['Attributes'] = atts
outDf2['Values'] = vals


In [ ]:
atts = []
vals = []

for (att, val) in zip(outDf2['Attributes'],outDf2['Values']):

    if att == 'nan' and val == None:
        continue
    elif '*****' in att and val == None:
        continue
    else:
        atts.append(att.replace('\t','').strip())
        vals.append(val.replace('\t','').strip())



In [ ]:
outDf3 = pd.DataFrame(columns =['Attributes', 'Values'])

outDf3['Attributes'] = atts
outDf3['Values'] = vals


In [ ]:
uniqueAtt = []

for att in outDf3['Attributes']:

    if att not in uniqueAtt:

        uniqueAtt.append(att)

In [ ]:
uniqueAtt

['Point System Name',
 'Point Name',
 'Point Type',
 'Supervised',
 'Revision Number',
 'Classification',
 'Descriptor',
 'Panel Name',
 'Auto Unbundled',
 'Modified from Default',
 'Point Address',
 'Actuator Type',
 'Slope',
 'COV Limit',
 'Engineering Units',
 'Analog Representation',
 '# of decimal places',
 'Initial Value',
 'Totalization',
 'Enabled for RENO',
 'Alarm Issue Management',
 'Graphic Name',
 'Informational Text',
 'Alarm Type',
 '4/26/2022                          Insight Job                          02',
 'Selection',
 'Text Table',
 'Sensor Type',
 '4/27/2022                          Insight Job                          09']

In [ ]:
outDf3

,Attributes,Values
0,Point System Name,AHU1.ZONE:ACT CLG STPT
1,Point Name,AHU1.ZONE:ACT CLG STPT
2,Point Type,LAO
3,Supervised,Yes
4,Revision Number,3
...,...,...
9144,Enabled for RENO,No
9145,Alarm Issue Management,No
9146,Graphic Name,<Undefined>
9147,Informational Text,<< No Text Defined >>


In [ ]:
sysNames = []
pointIndex = 0
pointIndexies = []
for count, i in enumerate(outDf3['Attributes']):
  if i == 'Point Name': #'Point System Name'
    sysNames.append(outDf3['Values'][count])
    pointIndexies.append(count)

pointIndexies.append(count)
print(sysNames)

['AHU1.ZONE:ACT CLG STPT', 'AHU1.ZONE:ACT HTG STPT', 'AHU1.ZONE:DESIRED OPER', 'AHU1.ZONE:EFF CLG STPT', 'AHU1.ZONE:EFF HTG STPT', 'AHU1.ZONE:MODE INPUT', 'AHU1.ZONE:NX OCC TIME', 'AHU1.ZONE:NX START IN', 'AHU1.ZONE:NX STOP IN', 'AHU1.ZONE:NX STOP TIME', 'AHU1.ZONE:NX STRT TIME', 'AHU1.ZONE:NX VAC IN', 'AHU1.ZONE:NX VAC TIME', 'AHU1.ZONE:OA TEMP', 'AHU1.ZONE:OA TEMP STOP', 'AHU1.ZONE:OA TEMP STRT', 'AHU1.ZONE:OCC CLG STPT', 'AHU1.ZONE:OCC HTG STPT', 'AHU1.ZONE:OCC TIME', 'AHU1.ZONE:PHASE', 'AHU1.ZONE:STOP DRTN', 'AHU1.ZONE:STOP MODE', 'AHU1.ZONE:STOP TIME', 'AHU1.ZONE:STRT DRTN', 'AHU1.ZONE:STRT MODE', 'AHU1.ZONE:STRT TIME', 'AHU1.ZONE:TEMP DV STOP', 'AHU1.ZONE:TEMP DV STRT', 'AHU1.ZONE:TG TEMP STOP', 'AHU1.ZONE:TG TEMP STRT', 'AHU1.ZONE:TIME DV STOP', 'AHU1.ZONE:TIME DV STRT', 'AHU1.ZONE:VAC CLG STPT', 'AHU1.ZONE:VAC HTG STPT', 'AHU1.ZONE:VAC TIME', 'AHU1.ZONE:ZN TEMP', 'AHU1.ZONE:ZN TEMP STOP', 'AHU1.ZONE:ZN TEMP STRT', 'CV121E:AIR VOLUME', 'CV121E:CTL STPT', 'CV121E:DMPR POS', 'CV12

In [ ]:
len(pointIndexies)

397

In [ ]:
dfCols = ['Point System Name', 'Point Type', 'Supervised', 'Revision Number', 'Classification', 'Descriptor', 'Panel Name', 'Auto Unbundled', 'Modified from Default', 'Point Address', 'Actuator Type', 'Slope', 'COV Limit', 'Engineering Units', 'Analog Representation', '# of decimal places', 'Initial Value', 'Totalization', 'Enabled for RENO', 'Alarm Issue Management', 'Graphic Name', 'Informational Text', 'Alarm Type', 'Text Table', 'Sensor Type']

#'Point System Name:', 'Point Name:', 'Point Type:','Descriptor:', 'Value:', 'Condition:', 'Priority:', 'Analog Representation:', '# of decimal places:', 'Engineering Units:', 'Access Group(s):', 'Alarm Type:', 'Totalization:', 'Field Panel:', ' Point Address:', 'Sensor Type:', 'Wire Resistance:', 'Slope:', 'Intercept:', 'COV Limit:', 'Text Table 1:', 'On/Off Point Address:', 'Inverted:', 'Proof Point Address:', 'Normally Closed:', 'Proof Delay:', 'Actuator Type', 'High Limit:', 'Low Limit:', 'Object Name:', 'Object ID:', 'BACnet Command Priority Array',	'Relinquish Default:', 'Notification Class:', 'Annunciate To-Normal:', 'Annunciate To-OffNorrmal:', 'Annunciate To-Fault:', 'Print Alarms on BLN:', 'Alarm Count 2:', 'Normal ack Enabled:'

fillDf = pd.DataFrame({'Point Name': sysNames})

In [ ]:
fillDf = fillDf.reindex(columns = fillDf.columns.tolist() + dfCols)

In [ ]:
fillDf.head(6)

,Point Name,Point System Name,Point Type,Supervised,Revision Number,Classification,Descriptor,Panel Name,Auto Unbundled,Modified from Default,...,# of decimal places,Initial Value,Totalization,Enabled for RENO,Alarm Issue Management,Graphic Name,Informational Text,Alarm Type,Text Table,Sensor Type
0,AHU1.ZONE:ACT CLG STPT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AHU1.ZONE:ACT HTG STPT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AHU1.ZONE:DESIRED OPER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AHU1.ZONE:EFF CLG STPT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AHU1.ZONE:EFF HTG STPT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AHU1.ZONE:MODE INPUT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#for count, sysName in enumerate(fillDf['Point System Name:']):


for count, sysName in enumerate(fillDf['Point Name']):

    for count2, (thisCol, thisVal) in enumerate(zip(outDf3['Attributes'], outDf3['Values'])):
        #print(thisCol,inDf['Value'][count2])

        #print(thisCol, thisVal)

        if count2 > pointIndexies[count] and count2 < pointIndexies[count+1]:

              try:

                  fillDf[thisCol][count] = outDf3['Values'][count2]

              except:

                  print(outDf3['Attributes'][count2], outDf3['Values'][count2])


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  from ipykernel import kernelapp as app
/usr/local/lib/python3.7/dist-packages/pandas/core/indexing.py:1732: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_block(indexer, value, name)


4/26/2022                          Insight Job                          02 25 PM
Selection AHU1.ZONE:ACT CLG STPT, AHU1.ZONE:ACT HTG STPT, ...
4/26/2022                          Insight Job                          02 25 PM
Selection AHU1.ZONE:ACT CLG STPT, AHU1.ZONE:ACT HTG STPT, ...
4/26/2022                          Insight Job                          02 25 PM
Selection AHU1.ZONE:ACT CLG STPT, AHU1.ZONE:ACT HTG STPT, ...
4/26/2022                          Insight Job                          02 25 PM
Selection AHU1.ZONE:ACT CLG STPT, AHU1.ZONE:ACT HTG STPT, ...
4/26/2022                          Insight Job                          02 25 PM
Selection AHU1.ZONE:ACT CLG STPT, AHU1.ZONE:ACT HTG STPT, ...
4/26/2022                          Insight Job                          02 25 PM
Selection AHU1.ZONE:ACT CLG STPT, AHU1.ZONE:ACT HTG STPT, ...
4/26/2022                          Insight Job                          02 25 PM
Selection AHU1.ZONE:ACT CLG STPT, AHU1.ZONE:ACT HTG STPT, ...

In [ ]:
fillDf.head(3)

,Point Name,Point System Name,Point Type,Supervised,Revision Number,Classification,Descriptor,Panel Name,Auto Unbundled,Modified from Default,...,# of decimal places,Initial Value,Totalization,Enabled for RENO,Alarm Issue Management,Graphic Name,Informational Text,Alarm Type,Text Table,Sensor Type
0,AHU1.ZONE:ACT CLG STPT,AHU1.ZONE:ACT HTG STPT,LAO,Yes,3.0,Building Automation,ACT CLG STPT,PXCM01 (1),Yes,No,...,2.0,86.0,None Initial Priority: NONE,No,No,<Undefined>,<< No Text Defined >>,Not Alarmable,NaN,NaN
1,AHU1.ZONE:ACT HTG STPT,AHU1.ZONE:DESIRED OPER,LAO,Yes,3.0,Building Automation,ACT HTG STPT,PXCM01 (1),Yes,No,...,2.0,53.6,None Initial Priority: NONE,No,No,<Undefined>,<< No Text Defined >>,Not Alarmable,NaN,NaN
2,AHU1.ZONE:DESIRED OPER,AHU1.ZONE:EFF CLG STPT,LENUM,Yes,3.0,Building Automation,DESIRED OPER,PXCM01 (1),Yes,No,...,NaN,NaN,NaN,NaN,No,<Undefined>,<< No Text Defined >>,Not Alarmable,"SSTO_OPERATION, 0 - NONE, 1 - HEATING, 2 - COO...",NaN


In [ ]:
fillDf.to_csv('out.csv')